<a href="https://colab.research.google.com/github/geethat0126-bit/AI_5SEM/blob/main/WEEK7_1BM24CS104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def parse_expression(s):
    """Recursively parses a standard logic string into nested lists."""
    s = s.replace(" ", "")
    if '(' not in s:
        return s
    idx = s.index('(')
    name = s[:idx]
    args = []
    current_arg = []
    paren_count = 0
    for char in s[idx+1:-1]:
        if char == '(':
            paren_count += 1
            current_arg.append(char)
        elif char == ')':
            paren_count -= 1
            current_arg.append(char)
        elif char == ',' and paren_count == 0:
            args.append(parse_expression("".join(current_arg)))
            current_arg = []
        else:
            current_arg.append(char)
    if current_arg:
        args.append(parse_expression("".join(current_arg)))
    return [name] + args

def occurs_in(var, expr):
    if var == expr:
        return True
    if isinstance(expr, list):
        return any(occurs_in(var, sub_expr) for sub_expr in expr)
    return False

def apply_subst(expr, subst):
    if isinstance(expr, str):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        return [apply_subst(item, subst) for item in expr]
    return expr

def compose_subst(subst1, subst2):
    result = {}
    for var, val in subst1.items():
        result[var] = apply_subst(val, subst2)
    for var, val in subst2.items():
        if var not in result:
            result[var] = val
    return result

def format_expression(expr):
    if isinstance(expr, str):
        return expr
    return f"{expr[0]}({', '.join(format_expression(arg) for arg in expr[1:])})"

def format_subst(subst):
    if not subst:
        return "NIL"
    return "{" + ", ".join(f"{k}/{format_expression(v)}" for k, v in subst.items()) + "}"

def unify_algorithm(expr1_str, expr2_str, variables):
    """Performs unification and returns the final MGU dictionary or a FAILURE string."""
    try:
        psi1_orig = parse_expression(expr1_str)
        psi2_orig = parse_expression(expr2_str)
    except Exception:
        return "ERROR: Invalid syntax structure in expressions."

    def is_var(x): return isinstance(x, str) and x in variables
    def is_const(x): return isinstance(x, str) and x not in variables and not isinstance(x, list) and '(' not in str(x)

    def unify_core(p1, p2):
        if is_var(p1) or is_const(p1) or is_var(p2) or is_const(p2):
            if p1 == p2:
                return {}
            elif is_var(p1):
                if occurs_in(p1, p2):
                    return f"FAILURE: Occurs-check prevents unification of variable '{p1}' with '{format_expression(p2)}'."
                return {p1: p2}
            elif is_var(p2):
                if occurs_in(p2, p1):
                    return f"FAILURE: Occurs-check prevents unification of variable '{p2}' with '{format_expression(p1)}'."
                return {p2: p1}
            else:
                return f"FAILURE: Constant mismatch. Cannot bind constant '{p1}' to constant '{p2}'."

        # FIXED: Compare head names (p1[0] vs p2[0]) instead of full list structures
        if p1[0] != p2[0]:
            return f"FAILURE: Mismatched function/predicate head names '{p1[0]}' and '{p2[0]}'."

        if len(p1) != len(p2):
            return f"FAILURE: Argument count mismatch between '{format_expression(p1)}' and '{format_expression(p2)}'."

        current_subst = {}

        for i in range(1, len(p1)):
            term1 = apply_subst(p1[i], current_subst)
            term2 = apply_subst(p2[i], current_subst)

            s = unify_core(term1, term2)
            if isinstance(s, str) and s.startswith("FAILURE"):
                return s

            if s != {}:
                current_subst = compose_subst(current_subst, s)

        return current_subst

    return unify_core(psi1_orig, psi2_orig)

# =====================================================================
# CONTINUOUS LOOP RUNNER
# =====================================================================
if __name__ == "__main__":
    print("==================================================")
    print("      INTERACTIVE UNIFICATION ENGINE LOOPER       ")
    print("==================================================")
    print("Instructions: Enter expressions below to unify them.")
    print("Type 'exit' in any prompt field to turn off the loop.\n")

    # Standard logic variables definition
    default_vars = {'x', 'y', 'z', 'w'}

    while True:
        print("\n--- New Unification Pair ---")
        expr1 = input("Enter Expression 1 (Ψ1): ").strip()
        if expr1.lower() == 'exit':
            print("Exiting program. Goodbye!")
            break

        expr2 = input("Enter Expression 2 (Ψ2): ").strip()
        if expr2.lower() == 'exit':
            print("Exiting program. Goodbye!")
            break

        result = unify_algorithm(expr1, expr2, default_vars)

        print("\n" + "-" * 50)
        if isinstance(result, str) and result.startswith("FAILURE"):
            print("STATUS: Unification Failed.")
            print(f"REASON: {result}")
        elif isinstance(result, str) and result.startswith("ERROR"):
            print("STATUS: Syntax Error.")
            print(f"DETAILS: {result}")
        else:
            print("STATUS: Unified Successfully.")
            print(f"REQUIRED MGU: {format_subst(result)}")
        print("-" * 50)


      INTERACTIVE UNIFICATION ENGINE LOOPER       
Instructions: Enter expressions below to unify them.
Type 'exit' in any prompt field to turn off the loop.


--- New Unification Pair ---
Enter Expression 1 (Ψ1): P(b,x,f(g(z)))
Enter Expression 2 (Ψ2): P(z,f(y),f(y))

--------------------------------------------------
STATUS: Unified Successfully.
REQUIRED MGU: {z/b, x/f(g(b)), y/g(b)}
--------------------------------------------------

--- New Unification Pair ---
Enter Expression 1 (Ψ1): Q(y,g(a,b))
Enter Expression 2 (Ψ2): Q(g(x,x),y)

--------------------------------------------------
STATUS: Unification Failed.
REASON: FAILURE: Constant mismatch. Cannot bind constant 'b' to constant 'a'.
--------------------------------------------------

--- New Unification Pair ---
Enter Expression 1 (Ψ1): P(a,b,b)
Enter Expression 2 (Ψ2): P(x,y,z)

--------------------------------------------------
STATUS: Unified Successfully.
REQUIRED MGU: {x/a, y/b, z/b}
------------------------------------